<a href="https://colab.research.google.com/github/seihoon/workeraction/blob/main/TFlite_HelloWorld.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np

In [2]:
# 입력 데이터와 정답 데이터
x_train = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=float)
y_train = np.array([2.0, 4.0, 6.0, 8.0, 10.0], dtype=float)

In [ ]:
# 모델 구조 정의 (뉴런 1개)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=[1])
])

# 컴파일 (컴퓨터가 채점할 기준 설정)
model.compile(optimizer='sgd', loss='mean_squared_error')

# 학습 시작 (500번 반복 학습)
model.fit(x_train, y_train, epochs=500, verbose=1) # verbose=0은 중간 과정을 숨깁니다.
print("학습 완료!")

In [ ]:
# 변환하기 전, 원래 모델이 일반 Python 환경에서 정답을 잘 맞추는지 확인합니다. (10을 넣으면 약 20이 나와야 합니다.)
# 10을 입력했을 때 결과 예측
# print("원래 모델 예측 (입력: 10):", model.predict([10.0]))

# [10.0] 대신 np.array([[10.0]]) 형태로 전달합니다.
test_input = np.array([[10.0]], dtype=np.float32)
print("원래 모델 예측 (입력: 10):", model.predict(test_input))


In [ ]:
# TFLite 모델로 변환 및 저장
# 스마트폰이나 임베디드 장치(라즈베리 파이 등)에서 가볍게 돌아가도록 TFLite 포맷으로 변환
# 1. 변환기(Converter) 생성
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. TFLite 모델로 변환
tflite_model = converter.convert()

# 3. 파일로 저장하기
with open('multiplier_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("TFLite 모델 변환 및 저장 완료!")

# 파일에 multiplier_model.tflite 파일이 생겼는지 확인

In [ ]:
# 변환된 TFLite 모델 실행해보기 (검증)
# .tflite 파일을 직접 로드하여 모바일 기기처럼 시뮬레이션 구동
# 1. TFLite 인터프리터 초기화
interpreter = tf.lite.Interpreter(model_path="multiplier_model.tflite")
interpreter.allocate_tensors()

# 2. 입출력 텐서 정보 가져오기 (리스트 형태로 반환됨)
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 3. 테스트 데이터 입력 (인덱스 0번의 'index' 키를 참조해야 합니다)
test_input = np.array([[10.0]], dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input) # [0] 추가

# 4. 모델 실행
interpreter.invoke()

# 5. 결과 확인 (출력도 마찬가지로 0번 인덱스를 참조합니다)
tflite_results = interpreter.get_tensor(output_details[0]['index']) # [0] 추가
print("TFLite 모델 예측 (입력: 10):", tflite_results)

# 원래 모델과 마찬가지로 20에 가까운 결과값 19.83이 출력되는 것을 확인

In [ ]:
# 다음 단계
# 이미지를 분류하는 조금 더 발전된 예제(MNIST 손글씨 인식)